In [8]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import randint, loguniform, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
SCORING = "f1_macro"
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/alaric/Desktop/studies/projects/ML/mental-health-risk-prediction


In [9]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze("columns")
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")
print(f"Training classes: {sorted(y_train.unique())}")

Training shape: (1600, 23)
Testing shape: (400, 23)
Training classes: [np.int64(0), np.int64(1), np.int64(2)]


In [10]:
logistic_search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE,
    ),
    param_grid={
        "C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "l1_ratio": [0.0, 1.0],
        "solver": ["saga"],
    },
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

random_forest_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions={
        "n_estimators": randint(200, 801),
        "max_depth": [None, 5, 10, 15, 20, 30],
        "min_samples_split": randint(2, 15),
        "min_samples_leaf": randint(1, 8),
        "max_features": ["sqrt", "log2", None],
        "class_weight": [None, "balanced", "balanced_subsample"],
    },
    n_iter=30,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

xgboost_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions={
        "n_estimators": randint(100, 801),
        "learning_rate": loguniform(0.01, 0.3),
        "max_depth": randint(3, 10),
        "min_child_weight": randint(1, 10),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "gamma": uniform(0.0, 0.5),
        "reg_alpha": loguniform(1e-8, 1.0),
        "reg_lambda": loguniform(0.1, 10.0),
    },
    n_iter=30,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)
searches = {
    "Logistic Regression": logistic_search,
    "Random Forest": random_forest_search,
    "XGBoost": xgboost_search,
}

In [11]:
for name, search in searches.items():
    print(f"Running {name} optimization...")
    search.fit(X_train, y_train)
    print(f"Best CV macro-F1: {search.best_score_:.4f}")
    print(f"Best parameters: {search.best_params_}\n")

Running Logistic Regression optimization...
Best CV macro-F1: 0.7507
Best parameters: {'C': 10.0, 'l1_ratio': 1.0, 'solver': 'saga'}

Running Random Forest optimization...
Best CV macro-F1: 0.6817
Best parameters: {'class_weight': None, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 8, 'n_estimators': 240}

Running XGBoost optimization...
Best CV macro-F1: 0.7033
Best parameters: {'colsample_bytree': np.float64(0.7103996728090174), 'gamma': np.float64(0.1481367528520412), 'learning_rate': np.float64(0.01754362390093243), 'max_depth': 3, 'min_child_weight': 7, 'n_estimators': 620, 'reg_alpha': np.float64(0.015064619068942013), 'reg_lambda': np.float64(0.24970737145052724), 'subsample': np.float64(0.602208846849441)}



In [12]:
cv_results = []
all_search_results = {}
for name, search in searches.items():
    all_results = pd.DataFrame(search.cv_results_)
    all_search_results[name] = all_results
    cv_results.append({
        "Model": name,
        "Best_CV_Macro_F1": search.best_score_,
        "CV_Std": all_results.loc[search.best_index_, "std_test_score"],
        "Best_Params": search.best_params_,
    })

optimization_summary = (
    pd.DataFrame(cv_results)
    .sort_values("Best_CV_Macro_F1", ascending=False)
    .reset_index(drop=True)
)
display(optimization_summary)
optimization_summary.to_csv(
    RESULTS_DIR / "hyperparameter_optimization_summary.csv",
    index=False,
)
for name, result in all_search_results.items():
    filename = name.lower().replace(" ", "_")
    result.to_csv(RESULTS_DIR / f"{filename}_search_results.csv", index=False)
best_search = searches[optimization_summary.iloc[0]["Model"]]
best_model_name = optimization_summary.iloc[0]["Model"]
best_model = best_search.best_estimator_
print(f"Selected model by CV macro-F1: {best_model_name}")

,Model,Best_CV_Macro_F1,CV_Std,Best_Params
0,Logistic Regression,0.750748,0.020685,"{'C': 10.0, 'l1_ratio': 1.0, 'solver': 'saga'}"
1,XGBoost,0.703316,0.025475,"{'colsample_bytree': 0.7103996728090174, 'gamm..."
2,Random Forest,0.681731,0.023483,"{'class_weight': None, 'max_depth': 10, 'max_f..."


Selected model by CV macro-F1: Logistic Regression


In [13]:
test_results = []
test_reports = {}
test_confusion_matrices = {}
class_labels = sorted(y_test.unique())
for name, search in searches.items():
    model = search.best_estimator_
    y_pred = model.predict(X_test)
    test_reports[name] = classification_report(
        y_test,
        y_pred,
        target_names=[str(label) for label in class_labels],
        zero_division=0,
        output_dict=True,
    )
    test_confusion_matrices[name] = pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=class_labels),
        index=[f"Actual {label}" for label in class_labels],
        columns=[f"Predicted {label}" for label in class_labels],
    )
    test_results.append({
        "Model": name,
        "CV_Macro_F1": search.best_score_,
        "Test_Accuracy": accuracy_score(y_test, y_pred),
        "Test_Precision_Macro": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Test_Recall_Macro": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Test_Macro_F1": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
    })
test_summary = (
    pd.DataFrame(test_results)
    .sort_values("Test_Macro_F1", ascending=False)
    .reset_index(drop=True)
)
display(test_summary)
test_summary.to_csv(RESULTS_DIR / "optimized_model_test_results.csv", index=False)
for name, report in test_reports.items():
    print(f"\n{name} class-specific recall:")
    print({label: values["recall"] for label, values in report.items()
           if label in {str(class_label) for class_label in class_labels}})
    print(test_confusion_matrices[name])

,Model,CV_Macro_F1,Test_Accuracy,Test_Precision_Macro,Test_Recall_Macro,Test_Macro_F1
0,Logistic Regression,0.750748,0.7525,0.751922,0.751783,0.751822
1,XGBoost,0.703316,0.7325,0.725889,0.731581,0.727460
2,Random Forest,0.681731,0.7125,0.708667,0.711824,0.710016



Logistic Regression class-specific recall:
{'0': 0.8235294117647058, '1': 0.803030303030303, '2': 0.6287878787878788}
          Predicted 0  Predicted 1  Predicted 2
Actual 0          112            0           24
Actual 1            1          106           25
Actual 2           25           24           83

Random Forest class-specific recall:
{'0': 0.7794117647058824, '1': 0.803030303030303, '2': 0.553030303030303}
          Predicted 0  Predicted 1  Predicted 2
Actual 0          106            1           29
Actual 1            3          106           23
Actual 2           30           29           73

XGBoost class-specific recall:
{'0': 0.8235294117647058, '1': 0.8257575757575758, '2': 0.5454545454545454}
          Predicted 0  Predicted 1  Predicted 2
Actual 0          112            0           24
Actual 1            3          109           20
Actual 2           31           29           72


In [14]:
for name, search in searches.items():
    filename = name.lower().replace(" ", "_")
    joblib.dump(search.best_estimator_, MODELS_DIR / f"{filename}_optimized.joblib")
    test_confusion_matrices[name].to_csv(
        RESULTS_DIR / f"{filename}_optimized_confusion_matrix.csv"
    )
joblib.dump(best_model, MODELS_DIR / "best_optimized_model.joblib")
print(f"Saved optimized models to {MODELS_DIR}")
print(f"Saved optimization results to {RESULTS_DIR}")

Saved optimized models to /home/alaric/Desktop/studies/projects/ML/mental-health-risk-prediction/models
Saved optimization results to /home/alaric/Desktop/studies/projects/ML/mental-health-risk-prediction/results
